# 🛟 Robô da Subdivisão de Meteorologia — plano B no Colab

Este notebook roda **os mesmos scripts do GitHub** (repo `meteorologia-hiremar`), só que lendo e gravando direto no seu Drive montado.
Use quando o GitHub Actions falhar ou quando quiser refazer algo na hora.

**Ordem do fechamento do mês:** 0 → 1 → 2 → 3 → 5. A célula 4 (DECEA) é para quando o PDF chegar.

No Colab não existe a proteção "editado à mão": aqui o que você mandar rodar **sobrescreve** os arquivos do mês. As células na planilha de Indicadores continuam protegidas: o robô só mexe nas que estão vazias ou que ele mesmo preencheu, a não ser que você marque `FORCAR = True`.

In [ ]:
# ============ 0) PREPARAÇÃO (rode sempre primeiro) ============
import os
from google.colab import drive
drive.mount('/content/drive')

# Mês a processar: 'AAAA-MM'. Deixe '' para o mês anterior.
MES = ''
# True = sobrescreve até valores que você digitou à mão na planilha de Indicadores
FORCAR = False

PASTA_IND  = '/content/drive/My Drive/FAB/Indicadores MET'
PASTA_CONS = '/content/drive/My Drive/FAB/Indicadores MET/CONSISTENCIA'

# Chave da REDEMET: guarde uma vez no 🔑 (Secrets) do Colab com o nome REDEMET_KEY
try:
    from google.colab import userdata
    os.environ['REDEMET_KEY'] = userdata.get('REDEMET_KEY')
except Exception:
    os.environ['REDEMET_KEY'] = ''   # <- ou cole a chave aqui entre as aspas

# Baixa (ou atualiza) os scripts do GitHub
if os.path.exists('/content/meteorologia-hiremar'):
    !git -C /content/meteorologia-hiremar pull -q
else:
    !git clone -q --depth 1 https://github.com/hiremar/meteorologia-hiremar /content/meteorologia-hiremar
%cd /content/meteorologia-hiremar
!pip -q install pdfplumber xlsxwriter

ARG_MES = f'--mes {MES}' if MES else ''
ARG_FORCAR = '--forcar' if FORCAR else ''
PASTAS = f'--local-consistencia "{PASTA_CONS}" --local-indicadores "{PASTA_IND}"'
print('✅ Pronto. Mês:', MES or 'anterior')

## 1) REDEMET: completa a planilha do mês + atrasos/ausências + SBSP AUTOMETAR
Rebusca o mês inteiro, dia a dia (leva alguns minutos). Gera `ATRASOS_METAR_SPECI_AAAA_MM_FINAL.xlsx` e `SBSP_AUTOMETAR_MESANO.xlsx` na pasta Indicadores MET.

In [ ]:
!python mensal_redemet.py $ARG_MES --forcar $PASTAS

## 2) Auditoria de consistência (ICA 105-15)
Gera `AUDITORIA_<MES><ANO> CONSISTÊNCIA.xlsx` na pasta CONSISTENCIA. **Atenção:** sobrescreve a auditoria do mês. Se você já fez ajustes nela, pule esta célula.

In [ ]:
!python auditoria_metar.py $ARG_MES $PASTAS

## 3) Planilha de Indicadores (D = mensagens, E = atrasos provisórios, F = erros)

In [ ]:
!python atualizar_indicadores_drive.py $ARG_MES $ARG_FORCAR $PASTAS

## 4) Relatório do DECEA (quando o PDF chegar)
Solte o PDF (nome com "CRCEA") na pasta Indicadores MET e rode. Coluna E = atrasadas + ausentes (METAR + SPECI); em SBSP, sem o período 02:01Z–08:59Z. Também gera `DECEA_<MES><ANO>_ATRASOS_AUSENCIAS.xlsx` e o dashboard.

In [ ]:
!python decea_relatorio.py $ARG_FORCAR $PASTAS

## 5) Dashboard HTML
Gera `IndicadorMET_<MES><ANO>.html`, lendo o .xlsx de verdade e calculando as porcentagens a partir de D, E e F.

In [ ]:
!python gerar_dashboard.py $PASTAS